# Categorical Search Implementation with Typesense

This notebook demonstrates how to set up and use Typesense for categorical search similar to the KPI search functionality.

## 1. Make sure Typesense server is running

Ensure your Typesense server is running. If not, you can start it with:

```bash
docker run -d --name=typesense-server -p 8108:8108 -v typesense-data:/data typesense/typesense:0.25.2 --data-dir /data --api-key=xyz --enable-cors
```

## 2. Create the Typesense collection for categories

In [ ]:
import requests
import json

# Load schema from file
with open('cat_schema.json', 'r') as f:
    schema = json.load(f)

# Create the collection
url = "http://localhost:8108/collections"
headers = {
    "X-TYPESENSE-API-KEY": "xyz",
    "Content-Type": "application/json"
}

# Try to delete the collection if it exists already
try:
    delete_response = requests.delete(f"{url}/category_index", headers=headers)
    print(f"Deleted existing collection: {delete_response.status_code}")
except Exception as e:
    print(f"Error deleting collection: {e}")

# Create the collection
response = requests.post(url, headers=headers, json=schema)
print(f"Created collection: {response.status_code}")
print(json.dumps(response.json(), indent=2))

## 3. Prepare and Import the Categorical Data

In [ ]:
import pandas as pd
import json

# Load the CSV data
df = pd.read_csv('data/All_cat_values.csv')

# Display the first few rows to understand the structure
print("First 5 rows of data:")
print(df.head())

# Add a sort_order field (1-based index)
df['sort_order'] = range(1, len(df) + 1)

# Convert the DataFrame to a list of dictionaries
data = df.to_dict(orient='records')

# Export the data as JSONL
jsonl_file = 'category_data.jsonl'
with open(jsonl_file, 'w') as f:
    for record in data:
        f.write(json.dumps(record) + '\n')

print(f"✅ JSONL file created: {jsonl_file}")

## 4. Import the Data into Typesense

In [ ]:
import requests
import json

# Import the documents
url = "http://localhost:8108/collections/category_index/documents/import?action=create"
headers = {
    "X-TYPESENSE-API-KEY": "xyz",
    "Content-Type": "application/json"
}

# Read the JSONL file
with open('category_data.jsonl', 'r') as f:
    data = f.read()

# Import the documents
response = requests.post(url, headers=headers, data=data)
print(f"Import response status: {response.status_code}")

# Count records imported successfully
if response.status_code == 200:
    lines = response.text.strip().split('\n')
    successful = sum(1 for line in lines if json.loads(line).get('success', False))
    print(f"Successfully imported {successful} out of {len(lines)} records")
else:
    print("Error importing data:")
    print(response.text)

## 5. Test a Simple Search Query

In [ ]:
import requests
import json

# Test search query
url = "http://localhost:8108/collections/category_index/documents/search"
headers = {
    "X-TYPESENSE-API-KEY": "xyz"
}

# Search for a sample value - adjust the query as needed
search_term = "IRON"  # Example search term
params = {
    "q": search_term,
    "query_by": "Value",
    "per_page": 5
}

response = requests.get(url, headers=headers, params=params)
data = response.json()

# Pretty-print the result
print(f"Search results for '{search_term}':\n")
print(json.dumps(data, indent=2))

# Display the matched values in a more readable format
if data.get('found', 0) > 0:
    print("\nMatched Categories:")
    for i, hit in enumerate(data['hits'], 1):
        doc = hit['document']
        print(f"{i}. Value: {doc['Value']}")
        print(f"   Table: {doc['Table_name']}")
        print(f"   Column: {doc['Column_name']}")
        print(f"   Path: {doc['Table_path']}")
        print()

## 6. Using the CategoryMatcher Class

In [ ]:
from category_matcher import CategoryMatcher

# Initialize the matcher
matcher = CategoryMatcher()

# Test with a query
query = "iron ferrous molecule"  # Example query
results = matcher.find_categories(query)

print(f"Search results for: '{query}'\n")
print(f"Found {len(results)} matches:")

for i, result in enumerate(results, 1):
    print(f"{i}. Value: {result['Value']}")
    print(f"   Table: {result['Table_name']}")
    print(f"   Column: {result['Column_name']}")
    print(f"   Path: {result['Table_path']}")
    print()

## 7. Advanced Usage - Faceted Search

In [ ]:
import requests
import json

# Faceted search example
url = "http://localhost:8108/collections/category_index/documents/search"
headers = {
    "X-TYPESENSE-API-KEY": "xyz"
}

# Search with faceting on Table_name
search_term = "IRON"  # Example search term
params = {
    "q": search_term,
    "query_by": "Value",
    "facet_by": "Table_name",
    "per_page": 10
}

response = requests.get(url, headers=headers, params=params)
data = response.json()

# Print facet counts
print(f"Facet counts for '{search_term}' by Table_name:\n")
if 'facet_counts' in data and data['facet_counts']:
    for facet in data['facet_counts']:
        print(f"Facet: {facet['field_name']}")
        for count in facet['counts']:
            print(f"  {count['value']}: {count['count']}")
else:
    print("No facet counts found.")